# 第 23 天：因子拥挤

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：因子拥挤
> 必做：相关性分析
> 选做：聚类分析
> 目标产出：因子相关矩阵

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 构造因子相关矩阵，识别重复信号。
2. 用简单聚类把相似因子归组。
3. 建立拥挤度评分：相关性、持仓重叠、换手压力一起看。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

一个电影院里，如果所有人都挤在同一个出口，风险就不在于出口不存在，而在于大家同时冲出去。因子拥挤研究的就是这种“信号太多人用”的状态。

## 5. 今日核心实验


### 实验 1：因子相关矩阵：先看声音是否重复

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
factor_panel = pd.concat({name: factor.stack() for name, factor in factor_library.items()}, axis=1)
corr_matrix = factor_panel.corr()
print(corr_matrix.round(2))


### 实验 2：热力图：让拥挤一眼可见

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index)
ax.set_title("因子相关矩阵")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
plt.close()


### 实验 3：简单聚类：相关性超过阈值就归为同一家族

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
threshold = 0.55
unassigned = set(corr_matrix.columns)
clusters = []

while unassigned:
    seed = sorted(unassigned)[0]
    group = {seed}
    for other in list(unassigned):
        if other != seed and abs(corr_matrix.loc[seed, other]) >= threshold:
            group.add(other)
    clusters.append(sorted(group))
    unassigned -= group

cluster_table = pd.DataFrame({
    "cluster": [i + 1 for i, group in enumerate(clusters) for _ in group],
    "factor": [factor for group in clusters for factor in group],
})
print(cluster_table)


### 实验 4：持仓重叠：相关性之外的拥挤证据

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
def top_overlap(a: pd.DataFrame, b: pd.DataFrame, q: float = 0.2) -> float:
    top_a = a.rank(axis=1, pct=True) >= 1 - q
    top_b = b.rank(axis=1, pct=True) >= 1 - q
    both = (top_a & top_b).sum(axis=1)
    union = (top_a | top_b).sum(axis=1).replace(0, np.nan)
    return (both / union).mean()

overlap = pd.DataFrame(index=factor_library.keys(), columns=factor_library.keys(), dtype=float)
for left in factor_library:
    for right in factor_library:
        overlap.loc[left, right] = top_overlap(factor_library[left], factor_library[right])

print(overlap.round(2))


### 实验 5：拥挤评分：相关、重叠、换手合在一起看

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
crowding_rows = []
for name, factor in factor_library.items():
    avg_abs_corr = corr_matrix[name].drop(name).abs().mean()
    avg_overlap = overlap[name].drop(name).mean()
    turnover = top_bucket_turnover(factor).mean()
    crowding_score = 0.45 * avg_abs_corr + 0.35 * avg_overlap + 0.20 * turnover
    crowding_rows.append({
        "factor": name,
        "avg_abs_corr": avg_abs_corr,
        "avg_top_overlap": avg_overlap,
        "turnover": turnover,
        "crowding_score": crowding_score,
    })

crowding = pd.DataFrame(crowding_rows).set_index("factor").sort_values("crowding_score", ascending=False)
print(crowding.round(4))


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：因子拥挤
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：只看因子值相关性，不看组合持仓重叠。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：看到高相关就全部删除，忽略不同市场状态下的互补性。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：用长期相关性掩盖近期突然拥挤。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：把拥挤等同于无效，实际它更多是风险管理信号。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 24 天开始 Barra 基础，把因子暴露、风格因子和行业因子放进统一框架。

## 13. 一句话收尾

因子拥挤 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本、严格样本外检验和风险约束。
